# 도시 보행자 통행량 예측 — 시계열 EDA와 모델 (Melbourne Pedestrian)

- 데이터: 멜버른 센서별 일별 보행자 수 (39,517행 · 10센서)
- 목표: 통행량 예측 — 다중 시계열
- 흐름: 불러오기 → 시계열 EDA → 형식 변환 → 학습 → 해석
- 참고: 데이터 소개 data_pedestrian_counts.txt

- 이 데이터의 핵심: **COVID 구조 단절**
  → "과거 패턴이 미래에 안 통하는" 상황을 직접 확인

## 1. 불러오기

- 이미 3열 형식(item_id · timestamp · target)
- 센서 10개 · 2009~2020년 4월

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("pedestrian_counts_daily_10sensors.csv",
                 parse_dates=["timestamp"])
print(df.shape)                    # (39517, 3)
print("센서:", df["item_id"].nunique())
df.head()

## 2. 시계열 EDA

- 순서가 핵심 → 시간축으로 본다
- 특히 2020년 3~4월의 급락에 주목

### 2-1. COVID 구조 단절 (핵심)

- 2020년 3월부터 봉쇄로 통행량 급락
- 한 센서(T9)의 월별 추이로 확인

In [ ]:
t9 = df[df["item_id"]=="T9"].set_index("timestamp")["target"]

t9.resample("MS").mean().plot(figsize=(12,3),
    title="T9 monthly mean - COVID drop at 2020")
plt.axvline(pd.Timestamp("2020-03-01"), color="red",
            linestyle="--", label="lockdown")
plt.legend()
plt.show()

# 2020년 2월 → 4월 급락 확인
print(t9.resample("MS").mean().loc["2020-01":"2020-04"].round(0))

### 2-2. 요일 주기

- 통행량은 요일에 따라 규칙적으로 변함
- 주중 vs 주말 차이 확인

In [ ]:
(df.assign(d=df["timestamp"].dt.day_name())
   .groupby("d")["target"].mean()
   .reindex(["Monday","Tuesday","Wednesday","Thursday",
             "Friday","Saturday","Sunday"])
   .plot(kind="bar", figsize=(8,3), title="mean by weekday"))
plt.show()
# 금요일 최다, 일요일 최소 → 주 단위 주기

### 2-3. 센서 간 스케일 차이

- 센서마다 통행량 규모가 크게 다름 (최대 12배)
- 전역 모델 학습 시 고려사항

In [ ]:
(df.groupby("item_id")["target"].mean()
   .sort_values(ascending=False)
   .plot(kind="bar", figsize=(8,3), title="mean by sensor"))
plt.show()
# T3(약 29000) vs T11(약 2400) → 12배 차이

## 3. 시계열 형식 변환

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame

ts = TimeSeriesDataFrame.from_data_frame(
    df, id_column="item_id", timestamp_column="timestamp")
ts = ts.convert_frequency(freq="D")   # 일 단위, 빈 날짜 채움
print("변환 완료:", ts.shape)

## 4. 학습 — 구조 단절 실험 (핵심)

- 이 실습의 핵심: "과거로 학습해 봉쇄 시점을 예측하면?"
- 같은 모델로 두 시점을 예측해 비교

### 4-1. 봉쇄 이전까지만 학습

- 2020년 1월까지 데이터로 학습
- 이후(2월·4월)를 예측

In [ ]:
from autogluon.timeseries import TimeSeriesPredictor

# 2020년 1월까지 학습 데이터
train_cut = ts.slice_by_time(
    start_time=ts.index.get_level_values("timestamp").min(),
    end_time=pd.Timestamp("2020-02-01"))

predictor = TimeSeriesPredictor(
    prediction_length=28,   # 약 4주
    target="target",
    freq="D",
).fit(train_cut, presets="medium_quality", time_limit=600)

In [ ]:
predictor.leaderboard()

## 5. 해석 — 예측 실패 확인

In [ ]:
predictions = predictor.predict(train_cut)

# T9 예측 vs 실제 (실제엔 봉쇄 급락이 있음)
predictor.plot(
    data=ts,
    predictions=predictions,
    item_ids=["T9"],
    max_history_length=90,
)
plt.show()

### 5-1. 무엇을 보는가 (강의자료 71p)

- 모델은 과거 패턴대로 "평소 통행량"을 예측했을 것
- 하지만 실제는 봉쇄로 급락 → 예측이 크게 빗나감
- **어떤 모델도 봉쇄를 예측할 수 없었다**
  → 예측 실패가 모델 탓인가? 데이터에 없는 사건 탓인가?

- 강의 53p 관광 수요 사례와 같은 구조
  (외부 사건은 과거 데이터만으로 예측 불가)

## 정리

- 이미 3열 형식 → 변환 간단
- 요일 주기 뚜렷 · 센서 간 스케일 12배 차이
- COVID 구조 단절 = 과거 패턴이 미래에 안 통함
- 봉쇄 이전 학습 → 봉쇄 예측 시 크게 빗나감
- 이것은 모델의 한계가 아니라 예측의 본질적 한계

- 시계열의 교훈: 데이터에 없는 미래 사건은 예측 못 한다
  → 외부 정보(정책·사건)를 함께 봐야 하는 이유